In [2]:
#!/usr/bin/env python3
"""
ChatGPT accuracy evaluation script.
Evaluates accuracy for different question types:
- "is used" questions: check for "yes"
- "is not used" questions: check for "no"
- "how many" questions: check match field
"""

import json
from tqdm import tqdm

# Load ChatGPT answers
json_file = '/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        chatgpt_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(chatgpt_data)} samples\n")

# Initialize counters for each category
is_used_correct = 0
is_used_total = 0

is_not_used_correct = 0
is_not_used_total = 0

how_many_correct = 0
how_many_total = 0

# Evaluate each item
for item in tqdm(chatgpt_data, desc="Evaluating answers"):
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    chatgpt_answer = item['chatgpt_answer'].lower()
    
    # Check for "is used" questions
    if "is used" in original_answer and "is not used" not in original_answer:
        is_used_total += 1
        if "yes" in chatgpt_answer:
            is_used_correct += 1
    
    # Check for "is not used" questions
    elif "is not used" in original_answer:
        is_not_used_total += 1
        if "no" in chatgpt_answer:
            is_not_used_correct += 1
    
    # Check for "how many" questions
    elif "how many" in question:
        how_many_total += 1
        if item.get('match') == 'MATCH':
            how_many_correct += 1

# Calculate accuracies
is_used_accuracy = (is_used_correct / is_used_total * 100) if is_used_total > 0 else 0
is_not_used_accuracy = (is_not_used_correct / is_not_used_total * 100) if is_not_used_total > 0 else 0
how_many_accuracy = (how_many_correct / how_many_total * 100) if how_many_total > 0 else 0

total_correct = is_used_correct + is_not_used_correct + how_many_correct
total_questions = is_used_total + is_not_used_total + how_many_total
overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0

# Print results
print(f"\n{'='*70}")
print(f"CHATGPT ACCURACY EVALUATION")
print(f"{'='*70}\n")

print(f"'IS USED' Questions:")
print(f"  Total: {is_used_total}")
print(f"  Correct (contains 'yes'): {is_used_correct}")
print(f"  Accuracy: {is_used_accuracy:.2f}%\n")

print(f"'IS NOT USED' Questions:")
print(f"  Total: {is_not_used_total}")
print(f"  Correct (contains 'no'): {is_not_used_correct}")
print(f"  Accuracy: {is_not_used_accuracy:.2f}%\n")

print(f"'HOW MANY' Questions:")
print(f"  Total: {how_many_total}")
print(f"  Correct (match field = MATCH): {how_many_correct}")
print(f"  Accuracy: {how_many_accuracy:.2f}%\n")

print(f"{'='*70}")
print(f"OVERALL ACCURACY")
print(f"{'='*70}")
print(f"Total questions: {total_questions}")
print(f"Total correct: {total_correct}")
print(f"Overall Accuracy: {overall_accuracy:.2f}%")
print(f"{'='*70}")

# Breakdown table
print(f"\nBREAKDOWN:")
print(f"{'Question Type':<20} {'Total':<10} {'Correct':<10} {'Accuracy':<10}")
print(f"-" * 50)
print(f"{'Is Used':<20} {is_used_total:<10} {is_used_correct:<10} {is_used_accuracy:.2f}%")
print(f"{'Is Not Used':<20} {is_not_used_total:<10} {is_not_used_correct:<10} {is_not_used_accuracy:.2f}%")
print(f"{'How Many':<20} {how_many_total:<10} {how_many_correct:<10} {how_many_accuracy:.2f}%")
print(f"-" * 50)
print(f"{'TOTAL':<20} {total_questions:<10} {total_correct:<10} {overall_accuracy:.2f}%")


Loading /home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers.json...
Loaded 7652 samples



Evaluating answers: 100%|██████████| 7652/7652 [00:00<00:00, 851231.02it/s]


CHATGPT ACCURACY EVALUATION

'IS USED' Questions:
  Total: 1402
  Correct (contains 'yes'): 813
  Accuracy: 57.99%

'IS NOT USED' Questions:
  Total: 5303
  Correct (contains 'no'): 3467
  Accuracy: 65.38%

'HOW MANY' Questions:
  Total: 947
  Correct (match field = MATCH): 328
  Accuracy: 34.64%

OVERALL ACCURACY
Total questions: 7652
Total correct: 4608
Overall Accuracy: 60.22%

BREAKDOWN:
Question Type        Total      Correct    Accuracy  
--------------------------------------------------
Is Used              1402       813        57.99%
Is Not Used          5303       3467       65.38%
How Many             947        328        34.64%
--------------------------------------------------
TOTAL                7652       4608       60.22%


In [3]:
#!/usr/bin/env python3
"""
ChatGPT grounded questions accuracy evaluation.
Calculates accuracy from match tags in grounded answers.
"""

import json
from tqdm import tqdm

# Load ChatGPT grounded accuracy results
json_file = '/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_grounded_accuracy.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        grounded_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(grounded_data)} samples\n")

# Count matches
total_qa = 0
match_count = 0
no_match_count = 0

for item in tqdm(grounded_data, desc="Evaluating grounded answers"):
    for qa in item.get('grounded_qa_with_matches', []):
        total_qa += 1
        if qa['match'] == 'MATCH':
            match_count += 1
        else:
            no_match_count += 1

# Calculate accuracy
accuracy = (match_count / total_qa * 100) if total_qa > 0 else 0

# Print results
print(f"\n{'='*60}")
print(f"CHATGPT GROUNDED QUESTIONS ACCURACY")
print(f"{'='*60}")
print(f"Total grounded QA pairs: {total_qa}")
print(f"Matched: {match_count}")
print(f"No Match: {no_match_count}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*60}")


Loading /home/as5606/projects/Hulu-Med/chatGPT/chatgpt_grounded_accuracy.json...
Loaded 1402 samples



Evaluating grounded answers: 100%|██████████| 1402/1402 [00:00<00:00, 830566.98it/s]


CHATGPT GROUNDED QUESTIONS ACCURACY
Total grounded QA pairs: 5599
Matched: 1244
No Match: 4355
Accuracy: 22.22%


In [ ]:
#!/usr/bin/env python3
"""
ChatGPT semantic variations consistency evaluation.
Calculates Consistency Score (CS) and Answer Variance Score (AVS).

CS = proportion of semantic variations answered correctly
AVS = average number of distinct answers per question
"""

import json
from tqdm import tqdm

# Load ChatGPT semantic variations answers
json_file = '/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        semantic_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(semantic_data)} samples\n")

# Calculate CS and AVS
cs_sum = 0  # Sum of per-item consistency scores
avs_sum = 0  # Sum of distinct answer counts per item

for item in tqdm(semantic_data, desc="Evaluating consistency"):
    variations = item.get('semantic_variations_with_matches', [])
    
    if len(variations) == 0:
        continue
    
    # Calculate per-item Consistency Score
    # CS_i = (1/K) * Σ 1[match = MATCH]
    matches = sum(1 for v in variations if v.get('match') == 'MATCH')
    K = len(variations)
    cs_per_item = matches / K
    cs_sum += cs_per_item
    
    # Calculate per-item Answer Variance Score
    # AVS_i = |distinct answers for this item|
    distinct_answers = set(v['chatgpt_answer'].lower().strip() for v in variations)
    avs_per_item = len(distinct_answers)
    avs_sum += avs_per_item

# Calculate final CS and AVS
num_items = len(semantic_data)
CS = cs_sum / num_items if num_items > 0 else 0
AVS = avs_sum / num_items if num_items > 0 else 0

# Print results
print(f"\n{'='*70}")
print(f"CHATGPT SEMANTIC VARIATIONS CONSISTENCY EVALUATION")
print(f"{'='*70}\n")

print(f"Consistency Score (CS): {CS:.4f}")
print(f"  Interpretation: {CS*100:.2f}% of semantic variations answered correctly")
print(f"  Perfect CS: 1.0 (all variations answered consistently)\n")

print(f"Answer Variance Score (AVS): {AVS:.4f}")
print(f"  Interpretation: Average {AVS:.2f} distinct answers per question")
print(f"  Perfect AVS: 1.0 (same answer for all variations)\n")

print(f"{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"Total questions: {num_items}")
if CS >= 0.8 and AVS <= 1.5:
    print(f"Status: ✅ GOOD consistency (CS >= 0.8 and AVS <= 1.5)")
elif CS >= 0.7 and AVS <= 2:
    print(f"Status: ⚠️  MODERATE consistency (CS >= 0.7 and AVS <= 2)")
else:
    print(f"Status: ❌ POOR consistency (needs improvement)")

print(f"\nIdeal target: CS = 1.0 (100% correct), AVS = 1.0 (no variance)")
print(f"Current scores: CS = {CS:.4f}, AVS = {AVS:.4f}")
print(f"{'='*70}")


In [4]:
#!/usr/bin/env python3
"""
ChatGPT semantic variations accuracy evaluation for "how many" questions.
Adds match tags to "how many" questions and overwrites the original JSON file.
"""

import json
import os
import re
from tqdm import tqdm
from openai import OpenAI

client = OpenAI(api_key="os.environ.get("OPENAI_API_KEY")")


# System prompt for "how many" questions
SYSTEM_PROMPT = """\
You are an answer-matching evaluator for surgical VQA counting questions.

Your task is to compare two answers about quantity and decide whether they express the same number.

Return only one of the following labels:

MATCH
NO_MATCH

Guidelines:
- Mark MATCH if both answers refer to the SAME QUANTITY, regardless of wording
- Mark MATCH for number variations: "2" matches "Two" or "Two tools are operating"
- Mark MATCH for variations: "1" matches "One" or "One tool" or "There is one tool"
- Mark MATCH for written forms: "3" matches "three" or "There are three tools"
- Mark MATCH even if one answer is more verbose: "2" matches "Two surgical instruments are visible"
- Mark MATCH for number words: "four" matches "4" matches "There are 4 tools operating"
- Mark NO_MATCH if the numbers differ: "2" vs "3"
- Mark NO_MATCH if one says "no tools" or "0" and the other says there are tools
- Ignore capitalization, punctuation, and extra wording

Input format:
Answer 1: <first answer>
Answer 2: <second answer>

Output format:
MATCH or NO_MATCH
"""

def compare_answers(answer1: str, answer2: str) -> str:
    """Compare two answers and return whether they match in meaning."""
    user_text = f"Answer 1: {answer1}\nAnswer 2: {answer2}"
    
    try:
        response = client.responses.create(
            model="gpt-5.4-nano",
            instructions=SYSTEM_PROMPT,
            input=user_text,
        )
        
        result = response.output_text.strip()
        
        # Extract MATCH or NO_MATCH from response
        if "MATCH" in result.upper():
            if "NO_MATCH" in result.upper():
                no_match_idx = result.upper().find("NO_MATCH")
                match_idx = result.upper().find("MATCH")
                if no_match_idx < match_idx:
                    return "NO_MATCH"
            return "MATCH"
        else:
            return "NO_MATCH"
    
    except Exception as e:
        print(f"Error comparing answers: {e}")
        return None

# Load semantic variations answers
json_file = '/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        semantic_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(semantic_data)} samples\n")

# Add match field for "how many" questions only
how_many_count = 0
matched_count = 0

for item in tqdm(semantic_data, desc="Adding match field for 'how many' questions"):
    original_question = item['original_question'].lower()
    
    # Only evaluate "how many" questions
    if "how many" in original_question:
        how_many_count += 1
        original_answer = item['original_answer']
        
        # Add match field to each semantic variation
        for variation in item.get('semantic_variations_with_chatgpt_answers', []):
            chatgpt_answer = variation['chatgpt_answer']
            
            # Compare answers
            match = compare_answers(original_answer, chatgpt_answer)
            variation['match'] = match
            
            if match == 'MATCH':
                matched_count += 1

# Overwrite original file with updated data
with open(json_file, 'w') as f:
    json.dump(semantic_data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated and saved {len(semantic_data)} items to {json_file}")

# Print summary
print(f"\n{'='*60}")
print(f"ChatGPT 'HOW MANY' SEMANTIC VARIATIONS EVALUATION")
print(f"{'='*60}")
print(f"Total 'how many' questions: {how_many_count}")
print(f"Matched variations: {matched_count}")
if how_many_count > 0:
    # Count total variations for "how many" questions
    total_variations = 0
    for item in semantic_data:
        if "how many" in item['original_question'].lower():
            total_variations += len(item.get('semantic_variations_with_chatgpt_answers', []))
    
    accuracy = (matched_count / total_variations * 100) if total_variations > 0 else 0
    print(f"Total variations: {total_variations}")
    print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*60}")


Loading /home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json...
Loaded 7652 samples



Adding match field for 'how many' questions: 100%|██████████| 7652/7652 [44:59<00:00,  2.83it/s]  


✓ Updated and saved 7652 items to /home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json

ChatGPT 'HOW MANY' SEMANTIC VARIATIONS EVALUATION
Total 'how many' questions: 947
Matched variations: 1586
Total variations: 4735
Accuracy: 33.50%


In [5]:
#!/usr/bin/env python3
"""
ChatGPT semantic variations consistency score evaluation.
Calculates Consistency Score (CS) based on:
- "is used"/"is not used" questions: check for "yes"/"no"
- "how many" questions: check match tag
"""

import json
from tqdm import tqdm

# Load ChatGPT semantic variations answers
json_file = '/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json'

print(f"Loading {json_file}...")
try:
    with open(json_file, 'r') as f:
        semantic_data = json.load(f)
except FileNotFoundError:
    print(f"Error: File not found: {json_file}")
    exit(1)

print(f"Loaded {len(semantic_data)} samples\n")

# Calculate Consistency Score
cs_sum = 0  # Sum of per-item consistency scores
total_items = 0

for item in tqdm(semantic_data, desc="Evaluating consistency"):
    original_question = item['original_question'].lower()
    original_answer = item['original_answer'].lower()
    variations = item.get('semantic_variations_with_chatgpt_answers', [])
    
    if len(variations) == 0:
        continue
    
    total_items += 1
    matches = 0
    K = len(variations)
    
    for variation in variations:
        chatgpt_answer = variation['chatgpt_answer'].lower()
        is_correct = False
        
        # Check for "is used" questions
        if "is used" in original_answer and "is not used" not in original_answer:
            if "yes" in chatgpt_answer:
                is_correct = True
        
        # Check for "is not used" questions
        elif "is not used" in original_answer:
            if "no" in chatgpt_answer:
                is_correct = True
        
        # Check for "how many" questions
        elif "how many" in original_question:
            if variation.get('match') == 'MATCH':
                is_correct = True
        
        if is_correct:
            matches += 1
    
    # Calculate per-item consistency: (1/K) * Σ matches
    cs_per_item = matches / K
    cs_sum += cs_per_item

# Calculate final Consistency Score
CS = cs_sum / total_items if total_items > 0 else 0

# Print results
print(f"\n{'='*70}")
print(f"CHATGPT SEMANTIC VARIATIONS CONSISTENCY SCORE")
print(f"{'='*70}\n")

print(f"Consistency Score (CS): {CS:.4f}")
print(f"  Interpretation: {CS*100:.2f}% of semantic variations answered correctly")
print(f"  Perfect CS: 1.0 (all variations answered consistently)\n")

print(f"{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"Total questions evaluated: {total_items}")
if CS >= 0.8:
    print(f"Status: ✅ EXCELLENT consistency (CS >= 0.8)")
elif CS >= 0.7:
    print(f"Status: ✅ GOOD consistency (CS >= 0.7)")
elif CS >= 0.6:
    print(f"Status: ⚠️  MODERATE consistency (CS >= 0.6)")
else:
    print(f"Status: ❌ POOR consistency (needs improvement)")

print(f"\nTarget: CS = 1.0 (100% correct)")
print(f"Current: CS = {CS:.4f} ({CS*100:.2f}%)")
print(f"{'='*70}")


Loading /home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json...
Loaded 7652 samples



Evaluating consistency: 100%|██████████| 7652/7652 [00:00<00:00, 210501.97it/s]


CHATGPT SEMANTIC VARIATIONS CONSISTENCY SCORE

Consistency Score (CS): 0.4438
  Interpretation: 44.38% of semantic variations answered correctly
  Perfect CS: 1.0 (all variations answered consistently)

SUMMARY
Total questions evaluated: 7652
Status: ❌ POOR consistency (needs improvement)

Target: CS = 1.0 (100% correct)
Current: CS = 0.4438 (44.38%)


In [6]:
#!/usr/bin/env python3
"""
Calculate Answer Variance Score (AVS) for ChatGPT semantic variations.
AVS measures consistency of answers across semantic variations.
"""

import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json', 'r') as f:
    data = json.load(f)

total_variance = 0

for item in tqdm(data):
    # Collect all answers for this item's variations
    answers = []

    for variation in item.get('semantic_variations_with_chatgpt_answers', []):
        chatgpt_answer = variation['chatgpt_answer'].lower().strip()
        answers.append(chatgpt_answer)

    # Count distinct answers
    distinct_answers = len(set(answers))
    total_variance += distinct_answers

# Calculate AVS
avs = total_variance / len(data)
print(f"\nAnswer Variance Score (AVS): {avs:.4f}")
print(f"Interpretation:")
print(f"  - AVS = 1: Perfect consistency (same answer for all rephrasings)")
print(f"  - AVS > 1: Instability (different answers for same question)")


100%|██████████| 7652/7652 [00:00<00:00, 229959.91it/s]


Answer Variance Score (AVS): 4.8104
Interpretation:
  - AVS = 1: Perfect consistency (same answer for all rephrasings)
  - AVS > 1: Instability (different answers for same question)


In [7]:
#!/usr/bin/env python3
"""
Calculate Abstention Rate (AR) for ChatGPT on hallucinatory (unanswerable) questions.
AR measures the proportion of unanswerable questions on which the model correctly abstains.
"""

import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_hallucinatory_answers.json', 'r') as f:
    data = json.load(f)

total_questions = 0
abstention_count = 0

for item in tqdm(data, desc="Checking abstention rate"):
    new_questions = item.get('new_questions_with_chatgpt_answers', [])
    
    for question in new_questions:
        chatgpt_answer = question['chatgpt_answer'].lower().strip()
        total_questions += 1
        
        # Check if answer is "i don't know" (with or without punctuation)
        if "i don't know" in chatgpt_answer or "i dont know" in chatgpt_answer:
            abstention_count += 1

# Calculate Abstention Rate
ar = abstention_count / total_questions if total_questions > 0 else 0

print(f"\n{'='*70}")
print(f"CHATGPT HALLUCINATORY QUESTIONS ABSTENTION RATE")
print(f"{'='*70}\n")

print(f"Total unanswerable questions: {total_questions}")
print(f"Abstentions (I don't know): {abstention_count}")
print(f"Abstention Rate (AR): {ar:.4f}")
print(f"  Percentage: {ar*100:.2f}%\n")

print(f"Interpretation:")
print(f"  - AR = 1.0: Perfect (all unanswerable questions abstained)")
print(f"  - AR = 0.0: Poor (no abstentions, likely hallucinating)")
print(f"  - Higher AR = Better model behavior on unanswerable questions")
print(f"\n{'='*70}")


Checking abstention rate: 100%|██████████| 7652/7652 [00:00<00:00, 861536.37it/s]


CHATGPT HALLUCINATORY QUESTIONS ABSTENTION RATE

Total unanswerable questions: 15303
Abstentions (I don't know): 14409
Abstention Rate (AR): 0.9416
  Percentage: 94.16%

Interpretation:
  - AR = 1.0: Perfect (all unanswerable questions abstained)
  - AR = 0.0: Poor (no abstentions, likely hallucinating)
  - Higher AR = Better model behavior on unanswerable questions



In [8]:
#!/usr/bin/env python3
"""
Calculate False Confidence Rate (FCR) for ChatGPT on answerable questions.
FCR measures the proportion of answerable questions on which the model incorrectly abstains.
Higher FCR = worse (model being overly cautious on answerable questions).
"""

import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers_prompt.json', 'r') as f:
    data = json.load(f)

total_questions = 0
abstain_count = 0

for item in tqdm(data, desc="Checking false confidence rate"):
    chatgpt_answer = item.get('chatgpt_answer')
    
    # Skip if answer is None
    if chatgpt_answer is None:
        continue
    
    chatgpt_answer = chatgpt_answer.lower().strip()
    total_questions += 1
    
    # Check if answer is "i don't know" (with or without punctuation)
    if "i don't know" in chatgpt_answer or "i dont know" in chatgpt_answer:
        abstain_count += 1

# Calculate False Confidence Rate
fcr = abstain_count / total_questions if total_questions > 0 else 0

print(f"\n{'='*70}")
print(f"CHATGPT ANSWERABLE QUESTIONS - FALSE CONFIDENCE RATE")
print(f"{'='*70}\n")

print(f"Total answerable questions: {total_questions}")
print(f"Incorrectly abstained (I don't know): {abstain_count}")
print(f"False Confidence Rate (FCR): {fcr:.4f}")
print(f"  Percentage: {fcr*100:.2f}%\n")

print(f"Interpretation:")
print(f"  - FCR = 0.0: Perfect (answered all questions that should be answered)")
print(f"  - FCR = 1.0: Worst (abstained on all answerable questions)")
print(f"  - Higher FCR = Worse model behavior (overly cautious)")
print(f"\n{'='*70}")


Checking false confidence rate: 100%|██████████| 7652/7652 [00:00<00:00, 1480456.40it/s]


CHATGPT ANSWERABLE QUESTIONS - FALSE CONFIDENCE RATE

Total answerable questions: 7652
Incorrectly abstained (I don't know): 7308
False Confidence Rate (FCR): 0.9550
  Percentage: 95.50%

Interpretation:
  - FCR = 0.0: Perfect (answered all questions that should be answered)
  - FCR = 1.0: Worst (abstained on all answerable questions)
  - Higher FCR = Worse model behavior (overly cautious)



In [9]:
#!/usr/bin/env python3
"""
Analyze the effect of prompts on ChatGPT answers.
Shows how many correct answers change to "I don't know" when prompted.
"""

import json
from tqdm import tqdm

# Load both files
with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers.json', 'r') as f:
    original_answers = json.load(f)

with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers_prompt.json', 'r') as f:
    prompt_answers = json.load(f)

# Create mapping of ID to prompt answer for quick lookup
prompt_map = {item['id']: item for item in prompt_answers}

# Count metrics
original_correct = 0
changed_to_idk = 0
total_questions = len(original_answers)

for item in tqdm(original_answers, desc="Analyzing prompt effect"):
    item_id = item['id']
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    chatgpt_answer = item['chatgpt_answer'].lower()
    
    # Determine if original answer was correct
    is_correct = False
    
    # Check for "how many" questions - use match field
    if "how many" in question:
        if item.get('match') == 'MATCH':
            is_correct = True
    
    # Check for "is used" / "is not used" questions
    elif "is used" in original_answer or "is not used" in original_answer:
        if "is not used" in original_answer:
            # Should say "no"
            if "no" in chatgpt_answer:
                is_correct = True
        elif "is used" in original_answer:
            # Should say "yes"
            if "yes" in chatgpt_answer:
                is_correct = True
    
    if is_correct:
        original_correct += 1
        
        # Check if same question in prompted version changed to "I don't know"
        if item_id in prompt_map:
            prompt_answer = prompt_map[item_id].get('chatgpt_answer', '').lower().strip()
            
            if "i don't know" in prompt_answer or "i dont know" in prompt_answer:
                changed_to_idk += 1

# Calculate accuracies
original_accuracy = (original_correct / total_questions) * 100
new_accuracy = ((original_correct - changed_to_idk) / total_questions) * 100
accuracy_drop = original_accuracy - new_accuracy

print(f"\n{'='*70}")
print(f"CHATGPT PROMPT EFFECT ANALYSIS")
print(f"{'='*70}\n")

print(f"Original accuracy: {original_accuracy:.2f}%")
print(f"Originally correct answers: {original_correct}/{total_questions}")
print(f"\nAnswers changed to 'I don't know' due to prompt: {changed_to_idk}")
print(f"New accuracy (after prompt): {new_accuracy:.2f}%")
print(f"Accuracy drop: {accuracy_drop:.2f}%\n")

print(f"{'='*70}")
print(f"Interpretation:")
print(f"  - Negative accuracy drop = Prompt made model more cautious")
print(f"  - Larger drop = Prompt significantly degrades model reliability")
print(f"{'='*70}")


Analyzing prompt effect: 100%|██████████| 7652/7652 [00:00<00:00, 590651.37it/s]


CHATGPT PROMPT EFFECT ANALYSIS

Original accuracy: 60.22%
Originally correct answers: 4608/7652

Answers changed to 'I don't know' due to prompt: 4386
New accuracy (after prompt): 2.90%
Accuracy drop: 57.32%

Interpretation:
  - Negative accuracy drop = Prompt made model more cautious
  - Larger drop = Prompt significantly degrades model reliability


In [12]:
#!/usr/bin/env python3
"""
ChatGPT accuracy by question type.
Breaks down performance by is_used, is_not_used, and how_many questions.
Checks if answer contains "yes"/"no" instead of exact match.
"""

import json
from tqdm import tqdm

with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers.json', 'r') as f:
    data = json.load(f)

# Initialize counters for each category
categories = {
    'is_used': {'correct': 0, 'total': 0, 'incorrect': []},
    'is_not_used': {'correct': 0, 'total': 0, 'incorrect': []},
    'how_many': {'correct': 0, 'total': 0, 'incorrect': []}
}

for item in tqdm(data):
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    chatgpt_answer = item['chatgpt_answer'].lower().strip()
    match = item.get('match', '').upper()
    
    is_correct = False
    category = None
    
    # Categorize by question type
    if "how many" in question:
        category = 'how_many'
        if match == 'MATCH':
            is_correct = True
    
    elif "is not used" in original_answer:
        category = 'is_not_used'
        if "no" in chatgpt_answer:
            is_correct = True
    
    elif "is used" in original_answer:
        category = 'is_used'
        if "yes" in chatgpt_answer:
            is_correct = True
    
    # Count results
    if category:
        categories[category]['total'] += 1
        if is_correct:
            categories[category]['correct'] += 1
        else:
            categories[category]['incorrect'].append({
                'question': item['question'],
                'original_answer': item['original_answer'],
                'chatgpt_answer': item['chatgpt_answer']
            })

# Print results
print(f"\n{'='*70}")
print(f"CHATGPT ACCURACY BY QUESTION TYPE")
print(f"{'='*70}\n")

total_correct = 0
total_questions = 0

for cat_name in ['is_used', 'is_not_used', 'how_many']:
    cat = categories[cat_name]
    accuracy = (cat['correct'] / cat['total'] * 100) if cat['total'] > 0 else 0
    
    print(f"{cat_name.upper().replace('_', ' ')}:")
    print(f"  Total: {cat['total']}")
    print(f"  Correct: {cat['correct']}")
    print(f"  Incorrect: {cat['total'] - cat['correct']}")
    print(f"  Accuracy: {accuracy:.2f}%")
    print()
    
    total_correct += cat['correct']
    total_questions += cat['total']

# Overall accuracy
overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0
print(f"{'='*70}")
print(f"OVERALL ACCURACY: {overall_accuracy:.2f}%")
print(f"{'='*70}\n")


100%|██████████| 7652/7652 [00:00<00:00, 592658.24it/s]


CHATGPT ACCURACY BY QUESTION TYPE

IS USED:
  Total: 1402
  Correct: 813
  Incorrect: 589
  Accuracy: 57.99%

IS NOT USED:
  Total: 5303
  Correct: 3467
  Incorrect: 1836
  Accuracy: 65.38%

HOW MANY:
  Total: 947
  Correct: 328
  Incorrect: 619
  Accuracy: 34.64%

OVERALL ACCURACY: 60.22%



In [13]:
#!/usr/bin/env python3
"""
Gemini hallucinatory questions accuracy by type.
Analyzes performance on temporal_future, false_premise, external_knowledge,
and non_perceivable_attribute questions.
"""

import json
from tqdm import tqdm

# Load the data
with open('/home/as5606/projects/Hulu-Med/gemini/gemini_hallucinatory_answers.json', 'r') as f:
    data = json.load(f)

# Initialize counters for each type
question_types = {
    'temporal_future': {'correct': 0, 'total': 0, 'failed': []},
    'false_premise': {'correct': 0, 'total': 0, 'failed': []},
    'external_knowledge': {'correct': 0, 'total': 0, 'failed': []},
    'non_perceivable_attribute': {'correct': 0, 'total': 0, 'failed': []}
}

abstention_patterns = [
    "i don't know",
    "i do not know",
    "don't know",
    "cannot answer",
    "cannot determine",
    "unable to",
]

for item in tqdm(data):
    for qa in item.get('new_questions_with_gemini_answers', []):
        q_type = qa.get('type', 'unknown')
        gemini_answer = qa.get('gemini_answer')
        
        # Skip if answer is None
        if gemini_answer is None:
            continue
        
        gemini_answer = gemini_answer.lower()
        
        # Only process known types
        if q_type not in question_types:
            continue
        
        question_types[q_type]['total'] += 1
        
        # Check if model correctly abstained
        is_abstained = any(pattern in gemini_answer for pattern in abstention_patterns)
        
        if is_abstained:
            question_types[q_type]['correct'] += 1
        else:
            question_types[q_type]['failed'].append({
                'id': item['id'],
                'question': qa['question'],
                'gemini_answer': qa['gemini_answer']
            })

# Print results
print(f"{'='*70}")
print(f"GEMINI HALLUCINATORY QUESTIONS ACCURACY BY TYPE")
print(f"{'='*70}\n")

total_correct = 0
total_questions = 0

for q_type in ['temporal_future', 'false_premise', 'external_knowledge', 'non_perceivable_attribute']:
    cat = question_types[q_type]
    accuracy = (cat['correct'] / cat['total'] * 100) if cat['total'] > 0 else 0
    
    print(f"{q_type.upper().replace('_', ' ')}:")
    print(f"  Total: {cat['total']}")
    print(f"  Correctly abstained (said 'I don't know'): {cat['correct']}")
    print(f"  Incorrect (hallucinated): {cat['total'] - cat['correct']}")
    print(f"  Accuracy: {accuracy:.2f}%")
    print()
    
    total_correct += cat['correct']
    total_questions += cat['total']

# Overall accuracy
overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0
print(f"{'='*70}")
print(f"OVERALL HALLUCINATORY ACCURACY: {overall_accuracy:.2f}%")
print(f"{'='*70}\n")


100%|██████████| 7648/7648 [00:00<00:00, 288220.14it/s]

GEMINI HALLUCINATORY QUESTIONS ACCURACY BY TYPE

TEMPORAL FUTURE:
  Total: 5757
  Correctly abstained (said 'I don't know'): 1565
  Incorrect (hallucinated): 4192
  Accuracy: 27.18%

FALSE PREMISE:
  Total: 5299
  Correctly abstained (said 'I don't know'): 1431
  Incorrect (hallucinated): 3868
  Accuracy: 27.01%

EXTERNAL KNOWLEDGE:
  Total: 229
  Correctly abstained (said 'I don't know'): 199
  Incorrect (hallucinated): 30
  Accuracy: 86.90%

NON PERCEIVABLE ATTRIBUTE:
  Total: 1401
  Correctly abstained (said 'I don't know'): 1401
  Incorrect (hallucinated): 0
  Accuracy: 100.00%

OVERALL HALLUCINATORY ACCURACY: 36.23%



In [21]:
#!/usr/bin/env python3
"""
ChatGPT accuracy analysis by equipment/tool.
Analyzes performance across test, grounded, hallucinatory, and semantic variation questions.
"""

import json
from tqdm import tqdm
import statistics

# ============= VARIABLES - CHANGE THESE =============
tool_name = "hook"  # Change this: "scissors", "grasper", "hook", etc.
# ====================================================

print("="*80)
print(f"CHATGPT ACCURACY FOR '{tool_name.upper()}'")
print("="*80 + "\n")

# ============================================================================
# 1. GROUNDED QUESTIONS
# ============================================================================
print("1. GROUNDED QUESTIONS")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_grounded_accuracy.json', 'r') as f:
        grounded_data = json.load(f)

    grounded_matches = 0
    grounded_total = 0
    grounded_items = []

    for item in tqdm(grounded_data, desc="Processing grounded"):
        if tool_name in item['original_question'].lower():
            grounded_items.append(item['id'])
            for qa in item.get('grounded_qa_with_matches', []):
                grounded_total += 1
                if qa['match'] == 'MATCH':
                    grounded_matches += 1

    grounded_accuracy = (grounded_matches / grounded_total * 100) if grounded_total > 0 else 0
    print(f"Items with '{tool_name}': {len(grounded_items)}")
    print(f"Total subquestions: {grounded_total}")
    print(f"Correct (MATCH): {grounded_matches}")
    print(f"Accuracy: {grounded_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    grounded_accuracy = grounded_matches = grounded_total = 0

# ============================================================================
# 2. HALLUCINATORY QUESTIONS
# ============================================================================
print("2. HALLUCINATORY QUESTIONS")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_hallucinatory_answers.json', 'r') as f:
        halluc_data = json.load(f)

    halluc_matches = 0
    halluc_total = 0
    halluc_items = []
    
    abstention_patterns = [
        "i don't know",
        "i do not know",
        "don't know",
        "cannot answer",
        "cannot determine",
        "unable to",
    ]

    for item in tqdm(halluc_data, desc="Processing hallucinatory"):
        if tool_name in item['original_question'].lower():
            halluc_items.append(item['id'])
            for qa in item.get('new_questions_with_chatgpt_answers', []):
                chatgpt_answer = qa.get('chatgpt_answer', '').lower()
                if chatgpt_answer:
                    halluc_total += 1
                    is_abstained = any(pattern in chatgpt_answer for pattern in abstention_patterns)
                    if is_abstained:
                        halluc_matches += 1

    halluc_accuracy = (halluc_matches / halluc_total * 100) if halluc_total > 0 else 0
    print(f"Items with '{tool_name}': {len(halluc_items)}")
    print(f"Total subquestions: {halluc_total}")
    print(f"Correct (abstained): {halluc_matches}")
    print(f"Accuracy: {halluc_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    halluc_accuracy = halluc_matches = halluc_total = 0

# ============================================================================
# 3. SEMANTIC VARIATIONS
# ============================================================================
print("3. SEMANTIC VARIATIONS")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_semantic_variations_answers.json', 'r') as f:
        semantic_data = json.load(f)

    semantic_matches = 0
    semantic_total = 0
    semantic_items = []

    for item in tqdm(semantic_data, desc="Processing semantic"):
        if tool_name in item['original_question'].lower():
            semantic_items.append(item['id'])
            original_answer = item['original_answer'].lower()
            
            for qa in item.get('semantic_variations_with_chatgpt_answers', []):
                question = qa.get('question', '').lower()
                chatgpt_answer = qa.get('chatgpt_answer', '').lower()
                match = qa.get('match', '').upper()
                
                semantic_total += 1
                is_correct = False
                
                # Check for "how many" questions - use match field
                if "how many" in question:
                    if match == 'MATCH':
                        is_correct = True
                
                # Check for "is used" / "is not used" questions
                elif "is not used" in original_answer:
                    if "no" in chatgpt_answer:
                        is_correct = True
                elif "is used" in original_answer:
                    if "yes" in chatgpt_answer:
                        is_correct = True
                
                if is_correct:
                    semantic_matches += 1

    semantic_accuracy = (semantic_matches / semantic_total * 100) if semantic_total > 0 else 0
    print(f"Items with '{tool_name}': {len(semantic_items)}")
    print(f"Total subquestions: {semantic_total}")
    print(f"Correct: {semantic_matches}")
    print(f"Accuracy: {semantic_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    semantic_accuracy = semantic_matches = semantic_total = 0

# ============================================================================
# 4. TEST ACCURACY (Original Questions)
# ============================================================================
print("4. TEST ACCURACY (Original Questions)")
print("-"*80)

try:
    with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers.json', 'r') as f:
        test_data = json.load(f)

    test_matches = 0
    test_total = 0
    test_items = []

    for item in tqdm(test_data, desc="Processing test"):
        if tool_name in item['question'].lower():
            test_items.append(item['id'])
            
            question = item['question'].lower()
            original_answer = item['original_answer'].lower()
            chatgpt_answer = item['chatgpt_answer'].lower().strip()
            match = item.get('match', '').upper()
            
            test_total += 1
            is_correct = False
            
            # Check for "how many" questions - use match field
            if "how many" in question:
                if match == 'MATCH':
                    is_correct = True
            
            # Check for "is used" / "is not used" questions
            elif "is not used" in original_answer:
                if "no" in chatgpt_answer:
                    is_correct = True
            elif "is used" in original_answer:
                if "yes" in chatgpt_answer:
                    is_correct = True
            
            if is_correct:
                test_matches += 1

    test_accuracy = (test_matches / test_total * 100) if test_total > 0 else 0
    print(f"Questions with '{tool_name}': {len(test_items)}")
    print(f"Total questions: {test_total}")
    print(f"Correct: {test_matches}")
    print(f"Accuracy: {test_accuracy:.2f}%\n")
except FileNotFoundError:
    print(f"File not found\n")
    test_accuracy = test_matches = test_total = 0

# ============================================================================
# SUMMARY
# ============================================================================
print("="*80)
print(f"SUMMARY - {tool_name.upper()} ACCURACY")
print("="*80)
print(f"Grounded Questions:      {grounded_accuracy:.2f}% ({grounded_matches}/{grounded_total})")
print(f"Hallucinatory Questions: {halluc_accuracy:.2f}% ({halluc_matches}/{halluc_total})")
print(f"Semantic Variations:     {semantic_accuracy:.2f}% ({semantic_matches}/{semantic_total})")
print(f"Test Accuracy:           {test_accuracy:.2f}% ({test_matches}/{test_total})")
print("="*80)

all_matches = grounded_matches + halluc_matches + semantic_matches + test_matches
all_total = grounded_total + halluc_total + semantic_total + test_total
overall_avg = (all_matches / all_total * 100) if all_total > 0 else 0
print(f"\nOVERALL AVERAGE: {overall_avg:.2f}% ({all_matches}/{all_total})")

# Calculate mean and standard deviation
accuracies = [grounded_accuracy, halluc_accuracy, semantic_accuracy, test_accuracy]
mean_accuracy = statistics.mean(accuracies)
std_dev = statistics.stdev(accuracies) if len(accuracies) > 1 else 0

print(f"\nMean Accuracy: {mean_accuracy:.2f}%")
print(f"Standard Deviation: {std_dev:.2f}%")
print("="*80 + "\n")


CHATGPT ACCURACY FOR 'HOOK'

1. GROUNDED QUESTIONS
--------------------------------------------------------------------------------


Processing grounded: 100%|██████████| 1402/1402 [00:00<00:00, 1170464.61it/s]

Items with 'hook': 518
Total subquestions: 2066
Correct (MATCH): 351
Accuracy: 16.99%

2. HALLUCINATORY QUESTIONS
--------------------------------------------------------------------------------


Processing hallucinatory: 100%|██████████| 7652/7652 [00:00<00:00, 1484427.83it/s]

Items with 'hook': 934
Total subquestions: 1868
Correct (abstained): 1751
Accuracy: 93.74%

3. SEMANTIC VARIATIONS
--------------------------------------------------------------------------------



Processing semantic: 100%|██████████| 7652/7652 [00:00<00:00, 799014.49it/s]


Items with 'hook': 934
Total subquestions: 4670
Correct: 2882
Accuracy: 61.71%

4. TEST ACCURACY (Original Questions)
--------------------------------------------------------------------------------


Processing test: 100%|██████████| 7652/7652 [00:00<00:00, 1796519.13it/s]

Questions with 'hook': 934
Total questions: 934
Correct: 573
Accuracy: 61.35%

SUMMARY - HOOK ACCURACY
Grounded Questions:      16.99% (351/2066)
Hallucinatory Questions: 93.74% (1751/1868)
Semantic Variations:     61.71% (2882/4670)
Test Accuracy:           61.35% (573/934)

OVERALL AVERAGE: 58.26% (5557/9538)

Mean Accuracy: 58.45%
Standard Deviation: 31.53%



In [22]:
#!/usr/bin/env python3
"""
Analyze the effect of prompts on ChatGPT answers by question category.
Shows accuracy before and after prompt for "is used/is not used" and "how many" questions.
"""

import json
from tqdm import tqdm

# Load both files
with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers.json', 'r') as f:
    original_answers = json.load(f)

with open('/home/as5606/projects/Hulu-Med/chatGPT/chatgpt_answers_prompt.json', 'r') as f:
    prompt_answers = json.load(f)

# Create mapping of ID to prompt answer for quick lookup
prompt_map = {item['id']: item for item in prompt_answers}

# Initialize counters for each category
categories = {
    'is_used_not_used': {
        'original_correct': 0,
        'changed_to_idk': 0,
        'total': 0
    },
    'how_many': {
        'original_correct': 0,
        'changed_to_idk': 0,
        'total': 0
    }
}

for item in tqdm(original_answers, desc="Analyzing prompt effect"):
    item_id = item['id']
    question = item['question'].lower()
    original_answer = item['original_answer'].lower()
    chatgpt_answer = item['chatgpt_answer'].lower()
    
    # Determine category and correctness
    category = None
    is_correct = False
    
    # Check for "how many" questions
    if "how many" in question:
        category = 'how_many'
        if item.get('match') == 'MATCH':
            is_correct = True
    
    # Check for "is used" / "is not used" questions
    elif "is used" in original_answer or "is not used" in original_answer:
        category = 'is_used_not_used'
        if "is not used" in original_answer:
            if "no" in chatgpt_answer:
                is_correct = True
        elif "is used" in original_answer:
            if "yes" in chatgpt_answer:
                is_correct = True
    
    # Process if in a known category
    if category:
        categories[category]['total'] += 1
        
        if is_correct:
            categories[category]['original_correct'] += 1
            
            # Check if same question in prompted version changed to "I don't know"
            if item_id in prompt_map:
                prompt_answer = prompt_map[item_id].get('chatgpt_answer', '').lower().strip()
                
                if "i don't know" in prompt_answer or "i dont know" in prompt_answer:
                    categories[category]['changed_to_idk'] += 1

# Calculate and print results
print(f"\n{'='*70}")
print(f"CHATGPT PROMPT EFFECT ANALYSIS BY CATEGORY")
print(f"{'='*70}\n")

for cat_name in ['is_used_not_used', 'how_many']:
    cat = categories[cat_name]
    total = cat['total']
    original_correct = cat['original_correct']
    changed_to_idk = cat['changed_to_idk']
    
    original_accuracy = (original_correct / total * 100) if total > 0 else 0
    new_correct = original_correct - changed_to_idk
    new_accuracy = (new_correct / total * 100) if total > 0 else 0
    accuracy_drop = original_accuracy - new_accuracy
    
    print(f"{cat_name.upper().replace('_', ' ')}:")
    print(f"  Total questions: {total}")
    print(f"  Original accuracy: {original_accuracy:.2f}% ({original_correct}/{total})")
    print(f"  Answers changed to 'I don't know': {changed_to_idk}")
    print(f"  New accuracy (after prompt): {new_accuracy:.2f}% ({new_correct}/{total})")
    print(f"  Accuracy drop: {accuracy_drop:.2f}%\n")

print(f"{'='*70}")
print(f"Interpretation:")
print(f"  - Negative accuracy drop = Prompt made model more cautious")
print(f"  - Larger drop = Prompt significantly degrades model reliability")
print(f"{'='*70}\n")


Analyzing prompt effect: 100%|██████████| 7652/7652 [00:00<00:00, 525412.36it/s]


CHATGPT PROMPT EFFECT ANALYSIS BY CATEGORY

IS USED NOT USED:
  Total questions: 6705
  Original accuracy: 63.83% (4280/6705)
  Answers changed to 'I don't know': 4066
  New accuracy (after prompt): 3.19% (214/6705)
  Accuracy drop: 60.64%

HOW MANY:
  Total questions: 947
  Original accuracy: 34.64% (328/947)
  Answers changed to 'I don't know': 320
  New accuracy (after prompt): 0.84% (8/947)
  Accuracy drop: 33.79%

Interpretation:
  - Negative accuracy drop = Prompt made model more cautious
  - Larger drop = Prompt significantly degrades model reliability

